# Practicum: Conditional Memory Recall with Masked Stochastic Attention

In [lecture L15a](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026/tree/main/lectures/week-15/L15a) and [lab L15b](https://github.com/varnerlab/CHEME-5820-Labs-Spring-2026/tree/main/labs/week-15/L15b) we built a **Spiking Hebbian Memory Network (H-Mem)** that stores associations in a synaptic matrix $\mathbf{W}^{\mathrm{assoc}}$ and recalls them via $\mathbf{v} = \mathbf{W}^{\mathrm{assoc}}\mathbf{k}$, where $\mathbf{k}$ is a key (question) vector and $\mathbf{v}$ is the recalled value (answer). 

The **modern Hopfield network** of [Ramsauer et al. (2020)](https://arxiv.org/abs/2008.02217) generalizes this: stored memories sit on the columns of a memory matrix $\mathbf{X}$ and recall becomes a single softmax-attention update $\mathbf{s} \leftarrow \mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s})$, with an (inverse) temperature $\beta$ that interpolates between Hebbian recall and soft averaging.

Adding Langevin noise to the modern-Hopfield update turns it into a training-free generative sampler, **Stochastic Attention (SA)**, used recently to generate small protein families ([Varner, 2026](https://arxiv.org/abs/2603.14717)) and synthetic patient cohorts ([Varner et al., 2026](https://arxiv.org/abs/2604.07557)) without any model training. We talked about __stochastic attention__ in [lecture L8a](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026/tree/main/lectures/week-8/L8a) and [lab L8b](https://github.com/varnerlab/CHEME-5820-Labs-Spring-2026/tree/main/labs/week-8/L8b).

In this practicum, we explore whether SA over a small Hopfield memory of the [Olivetti faces](https://scikit-learn.org/stable/datasets/real_world.html#the-olivetti-faces-dataset) (10 subjects × 10 portraits per subject) acts as a training-free generator of recognizable *new* faces: blends of stored portraits, not copies. We then ask whether a single boolean mask (similar to the idea we used in decoder-only transformers) on the attention softmax is enough to steer that generator to a chosen subject, with no retraining and no change to the architecture.

> __Learning Objectives:__
>
> By the end of this practicum, you should be able to:
> * __Sample from a Hopfield memory with Stochastic Attention:__ Add Langevin noise to the modern-Hopfield update and treat the result as a training-free sampler from the network's energy landscape. Tune the step size, noise scale, and inverse temperature to balance fidelity against novelty in the generated faces.
> * __Steer generation by masking the attention softmax:__ Apply a boolean mask or a finite logit bias to the attention weights to restrict generation to a chosen subset of stored memories. Compare the two strategies and identify the calibration gap between attention-space conditioning and decoded output.
> * __Recover the Hebbian–Hopfield bridge empirically:__ Sweep the inverse temperature inside the masked sampler and read off the transition from soft blend to Hebbian recall in the novelty curve. Connect the high-temperature limit back to the H-Mem read step from [L15](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026/tree/main/lectures/week-15).

Let's get started!
___

## Background: From H-Mem to modern Hopfield to Stochastic Attention
The same associative-memory mechanism shows up multiple times across this course.

In [lecture L15a](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026/tree/main/lectures/week-15/L15a)/[lab L15b](https://github.com/varnerlab/CHEME-5820-Labs-Spring-2026/tree/main/labs/week-15/L15b) we saw the **spiking H-Mem** of Limbacher and Legenstein ([2020](https://proceedings.neurips.cc/paper/2020/file/f6876a9f998f6472cc26708e27444456-Paper.pdf), [2022/2023](https://arxiv.org/abs/2205.11276)). Memories are written into a synaptic matrix $\mathbf{W}^{\mathrm{assoc}}$ by Hebbian plasticity (the spike-trace outer-product rule), and a key spike train is recalled as $\mathbf{v} = \mathbf{W}^{\mathrm{assoc}}\mathbf{k}$, where $\mathbf{k}$ is the key and $\mathbf{v}$ is the recalled value. 

The **modern Hopfield network** of [Ramsauer et al. (2020)](https://arxiv.org/abs/2008.02217) replaces $\mathbf{W}^{\mathrm{assoc}}$ with the explicit memory matrix $\mathbf{X}$ (one stored pattern per column) and replaces the matrix–vector recall by one closed-form softmax-attention update for query $\mathbf{s}$ given by:
$$\mathbf{s} \;\leftarrow\; \mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s}).$$
In the limit $\beta \to \infty$ the softmax becomes a one-hot, and the update is exactly the Hebbian read step we used in [L15](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026/tree/main/lectures/week-15).

**Stochastic Attention (SA)** of [Varner (2026)](https://arxiv.org/abs/2603.14717) treats the modern-Hopfield log-sum-exp energy as a Boltzmann distribution and samples from it by Langevin dynamics, giving the update rule:
$$\mathbf{s}_{t+1} \;=\; (1-\eta)\,\mathbf{s}_{t} + \eta\cdot\mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s}_{t}) \;+\; \sigma\cdot\boldsymbol{\xi}_{t},\qquad \boldsymbol{\xi}_{t}\sim\mathcal{N}(0, \mathbf{I}).$$
where $\eta$ is a step size, $\sigma$ is a noise scale, $\boldsymbol{\xi}_{t}$ is a standard Gaussian noise vector, and $\beta$ is the inverse temperature, $\mathbf{s}_{t}$ is the current sample, $\mathbf{s}_{t+1}$ is the next sample and $\mathbf{X}$ is the memory matrix.
With $\sigma = 0$ and $\eta = 1$ this is the modern-Hopfield update; with $\sigma > 0$ stochastic attention draws novel samples from a smoothed family, training-free.

Let's take this a step further: can we steer the SA generator to a chosen subset of stored memories by masking the attention softmax, with no retraining and no change to the architecture? This is the question we explore in this practicum.

### Why a mask?
Stochastic Attention treats every stored memory equally. To restrict generation to a subset of memories, we add a boolean mask to the softmax: set the logit of every excluded pattern to $-\infty$. This is the masking idea from causal language modeling in transformers, applied to a Hopfield memory rather than a sequence. 

[Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115) showed that a softer version, a finite scalar bias on in-subset logits, also works, and characterized the **calibration gap** between what the sampler steers toward in attention space and what the decoded output looks like.

In this practicum we implement and explore both versions on the Olivetti faces dataset, verify the equivalence of the H-Mem read step and the modern-Hopfield update, and tune the SA hyperparameters to balance fidelity against novelty.
___

## Setup, Data, and Prerequisites

We set up the computational environment by including the [`Include.jl`](Include.jl) file, which sets paths, loads external packages, seeds the RNG, and pulls in the practicum source files from `src/`.

* [`src/Types.jl`](src/Types.jl) defines [`MyModernHopfieldNetworkModel`](src/Types.jl) and [`MyStochasticAttentionModel`](src/Types.jl).
* [`src/Factory.jl`](src/Factory.jl) provides the [`build(...)`](src/Factory.jl) factories.
* [`src/Compute.jl`](src/Compute.jl) implements [`modern_hopfield_recall`](src/Compute.jl), [`stochastic_attention_sample`](src/Compute.jl) (with optional `hard_mask` and `soft_bias` keyword arguments), and [`classify_by_nearest_mean`](src/Compute.jl).
* [`src/Files.jl`](src/Files.jl) ships [`load_olivetti_subset`](src/Files.jl) and [`image_grid`](src/Files.jl).

Let's load the environment.

In [1]:
include("Include.jl"); # load packages, src/ files, set random seed

  Activating project at `~/Desktop/practicum-5820-s2026-ElizabethSharon-1`
   Installed CodecZstd ────── v0.8.7
   Installed OpenEXR_jll ──── v3.4.9+0
   Installed FFTW_jll ─────── v3.3.12+0
   Installed IntervalSets ─── v0.7.14
   Installed DataStructures ─ v0.19.4
   Installed TiffImages ───── v0.11.9
    Updating `~/Desktop/practicum-5820-s2026-ElizabethSharon-1/Project.toml`
  [336ed68f] + CSV v0.10.16
  [5ae59095] + Colors v0.13.1
  [a93c6f00] + DataFrames v1.8.2
  [5789e2e9] + FileIO v1.18.0
  [82e4d734] + ImageIO v0.6.9
  [916415d5] + Images v0.26.2
  [872c559c] + NNlib v0.9.34
  [91a5bcdd] + Plots v1.41.6
  [08abe8d2] + PrettyTables v3.3.2
  [10745b16] + Statistics v1.11.1
  [2913bbd2] + StatsBase v0.34.10
  [37e2e46d] ~ LinearAlgebra ⇒ v1.11.0
  [9a3f8284] ~ Random ⇒ v1.11.0
  [8dfed614] ~ Test ⇒ v1.11.0
    Updating `~/Desktop/practicum-5820-s2026-ElizabethSharon-1/Manifest.toml`
  [621f4979] + AbstractFFTs v1.5.0
  [1520ce14] + AbstractTrees v0.4.5
  [79e6a3ab] + Adapt v4.5.

### Constants
Let's set some constants that define the problem size and dataset parameters. The comment next to each constant describes its purpose and values:

In [ ]:
# ── Setup: hyperparameters for the whole practicum ─────────────────────────
# STUDENT TASK: Uncomment the block below, then tweak the TODO knobs.
# We unpack everything in a `let` block so each binding is created exactly once
# and returned together as a tuple. Touch a knob, re-run this cell, and every
# downstream cell that references these names picks up the new value.

n_subjects, images_per_subject, β_gen, β_cond, β_soft = let

    # initialize -
    n_subjects = nothing;          # TODO: how many of the 40 Olivetti subjects to memorize (Int 1..40)
    images_per_subject = nothing;  # TODO: how many portraits per subject (Int 1..10)

    # β controls the sharpness of the softmax inside Stochastic Attention.
    # Large β → near-one-hot weights (snap onto a single stored portrait).
    # Small β → uniform weights (blended/mean state).
    β_gen  = nothing; # TODO: SA temperature for Task 1 (unconditional generation). Try 0.01, 0.02, 0.05.
    β_cond = nothing; # TODO: SA temperature for the hard-mask sampler (Task 2a) — large so the chain locks onto in-class memories.
    β_soft = nothing; # TODO: SA temperature for the log-multiplicity bias sampler (Task 2b). Picked so β·X'·s logit range Δ is comparable to log(ρ_max). Try 0.01, 0.02, 0.05.

    # return all five as a tuple.
    n_subjects, images_per_subject, β_gen, β_cond, β_soft;
end;

### Data

The [Olivetti faces dataset](https://scikit-learn.org/stable/datasets/real_world.html#the-olivetti-faces-dataset) has 40 subjects × 10 portraits each, $64 \times 64$ pixel grayscale. We've downloaded them and stored the files in the `data/olivetti/{faces,targets}.csv` in this repository. 

The helper function [`load_olivetti_subset(...)`](src/Files.jl) reads `images_per_subject::Int` portraits from each of the first `n_subjects::Int` people, flattens each image into a length-$4096$ vector, and stacks them as columns of a single memory matrix $\mathbf{X}$. The companion vector $\mathbf{y}$ records the integer subject id of each column.

The cell below loads the data into `X::Matrix{Float64}` and `y::Vector{Int}`.

In [ ]:
# STUDENT TASK: Uncomment the block below to load the Olivetti faces.
# Loads `n_subjects` subjects × `images_per_subject` portraits from the Olivetti
# face dataset. `X` is a (4096, N) matrix — one column per portrait, each column
# is a flattened 64×64 grayscale image with pixel values in [0, 1]. `y` is a
# length-N vector of subject ids in 0..n_subjects-1.

#==
X, y = load_olivetti_subset(images_per_subject; n_subjects = n_subjects);
println("loaded $(size(X, 2)) faces of dimension $(size(X, 1)) covering $(length(unique(y))) subjects")
==#

In [ ]:
y

__Test__: Before continuing, lets confirm that we have a $4096 \times N$ memory matrix with one subject id per column and `n_subjects::Int` distinct subjects present. If any of these checks fail, an [AssertionError](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) will be raised, and you should go back and check the data loading step.

In [ ]:
# Shape sanity checks. If any of these fail, every downstream cell will be
# wrong, so we want a loud failure here rather than a silent surprise later.
let
    @assert size(X, 1) == 4096                                # 64 × 64 pixels per face, flattened column-major
    @assert size(X, 2) == n_subjects * images_per_subject     # one column per portrait, total = n_subjects × images_per_subject
    @assert length(y) == size(X, 2)                           # one subject id per column
    @assert sort(unique(y)) == collect(0:(n_subjects - 1))    # the first n_subjects subjects, 0..n_subjects-1
end

__Did we blow up?__ If not, let's peek at one portrait of every subject we loaded, to confirm the data made it in correctly. If everything looks good, we should see a grid of 10 faces, each from a different subject.

In [ ]:
# Preview: one portrait per subject, laid out in a balanced grid with per-tile subject ids.
# This is sanity-checking the data, not part of the model — if the faces look like
# faces and the subject ids are correctly aligned with the images, we're good.
let
    ncols = 5
    nrows = cld(n_subjects, ncols) # ceil-division so every subject gets a tile
    plots = [];

    # One tile per subject: pick the FIRST stored portrait of that subject.
    for s in 0:(n_subjects - 1)
        idx = findfirst(==(s), y);                                    # column index of the first portrait of subject s
        img  = Gray.(reshape(X[:, idx], 64, 64));                     # un-flatten into a 64×64 grayscale image
        push!(plots, heatmap(img, color = :grays, axis = false, ticks = false,
            title = "subjid: $s", titlefontsize = 8, aspect_ratio = :equal));
    end

    # Pad the grid with empty plots so the layout is rectangular even when
    # n_subjects is not a multiple of ncols (e.g. 10 subjects in a 3×5 grid).
    for _ in (n_subjects + 1):(nrows * ncols)
        push!(plots, plot(framestyle = :none));
    end

    # Render the grid.
    plot(plots...; layout = (nrows, ncols),
        size = (150 * ncols, 160 * nrows),
        margin = 0Plots.PlotMeasures.mm)
end

### Build the SA models and reference classifier
Next, let's construct three objects for reuse throughout the notebook:
* `sa_gen::MyStochasticAttentionModel`: SA sampler at $\beta_{\mathrm{gen}}$ for unconditional generation (Task 1).
* `sa_cond::MyStochasticAttentionModel`: SA sampler at $\beta_{\mathrm{cond}}$ for **hard-mask** conditional generation (Task 2a). High $\beta$ so the chain locks onto in-class stored portraits.
* `sa_soft::MyStochasticAttentionModel`: SA sampler at $\beta_{\mathrm{soft}}$ for **log-multiplicity bias** conditional generation (Task 2b). Low $\beta$ so $\log\rho$ can compete with the $\beta\cdot\mathbf{X}^{\top}\mathbf{s}$ logit range; the optimal $\beta$ here differs from `sa_cond`, which is the calibration-gap regime of [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115).
* `class_means::Dict{Int, Vector{Float64}}`: per-subject centroids of the stored portraits, used as a training-free reference classifier in Task 2. This is computed by [`build_class_means(X, y)` function](src/Compute.jl), which groups the columns of $\mathbf{X}$ by their subject id in $\mathbf{y}$ and averages within each group: $\mathbf{c}_k = \tfrac{1}{|\{j : y_j = k\}|}\sum_{j : y_j = k}\mathbf{X}_{:,j}$. The result is one length-$4096$ mean vector per unique subject id.

The [`build(...)` factories](src/Factory.jl) for the SA samplers live in [`src/Factory.jl`](src/Factory.jl); [`build_class_means`](src/Compute.jl) lives in [`src/Compute.jl`](src/Compute.jl).

In [ ]:
# ── Build the three Stochastic Attention samplers and the per-class means ──
# STUDENT TASK: Uncomment the block below. Tweak step_size / noise_scale TODOs
# if you want to explore — defaults are reasonable.
# We need three different SA models because each task uses a different β.
# Building them once here means later cells can just call `stochastic_attention_sample`
# without re-constructing the model on every call.

sa_gen, sa_cond, sa_soft, class_means = let

    # initialize - we set these to `nothing` so the build calls below fill them in.
    sa_gen      = nothing; # Task 1 — unconditional generation (low β, soft blend)
    sa_cond     = nothing; # Task 2a — hard-mask conditional (high β, in-class lock-on)
    sa_soft     = nothing; # Task 2b — log-multiplicity bias (low β so log(ρ) can compete)
    class_means = nothing; # one mean portrait per subject

    #==
    # Task 1 sampler — low β so the softmax is soft and the chain blends memories.
    sa_gen = build(MyStochasticAttentionModel, (
        memories     = X,
        labels       = y,
        β            = β_gen,
        step_size    = 1.0,  # TODO: try 0.25, 0.5, 1.0
        noise_scale  = 0.10, # TODO: try 0.02 (regurgitation), 0.10, 0.30
    ));

    # Task 2a sampler — high β so attention is near one-hot. Combined with the
    # hard mask the chain locks onto in-class memories.
    sa_cond = build(MyStochasticAttentionModel, (
        memories     = X,
        labels       = y,
        β            = β_cond,
        step_size    = 1.0,
        noise_scale  = 0.10,
    ));

    # Task 2b sampler — low β so log(ρ) (the soft-bias logit shift) can compete
    # with the data term β·X'·s. If β is too large the soft bias gets steam-rolled.
    sa_soft = build(MyStochasticAttentionModel, (
        memories     = X,
        labels       = y,
        β            = β_soft,
        step_size    = 1.0,
        noise_scale  = 0.10,
    ));

    # Per-subject mean portraits, used by the nearest-centroid classifier.
    class_means = build_class_means(X, y);
    ==#

    # return -
    sa_gen, sa_cond, sa_soft, class_means;
end;

___

## Task 1: Stochastic Attention sampling on faces
In this task, we ask whether adding a Gaussian-noise term to the modern-Hopfield update, no training, no GPU, no learned weights, turns it into a generative model that produces new faces we never stored. 

The modern-Hopfield update returns one of the stored columns of $\mathbf{X}$ (in the high-$\beta$ limit, exactly one; at moderate $\beta$, a softmax mixture sharply concentrated on one). A pure recall loop cannot generate anything new. Stochastic Attention adds a Gaussian noise term so the update samples from a Boltzmann distribution rather than collapsing to a stored memory:

$$\underbrace{\mathbf{s}_{t+1} \;=\; (1-\eta)\,\mathbf{s}_{t} + \eta\cdot\mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s}_{t})}_{\text{modern-Hopfield update}} \;+\; \underbrace{\sigma\cdot\boldsymbol{\xi}_{t}}_{\text{the new term}},\qquad \boldsymbol{\xi}_{t}\sim\mathcal{N}(0, \mathbf{I}).$$

The two knobs we tune in this task are the step size $\eta$ (`step_size`) and the noise scale $\sigma$ (`noise_scale`): together they trade fidelity (samples close to stored portraits) against novelty (samples that differ from any single stored portrait). See the [L8a lecture notes](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026/tree/main/lectures/week-8/L8a) for a derivation.

The cell below picks 10 random stored portraits as starting points (each with a small Gaussian nudge so chains do not start identically), then iterates the SA update 3000 times to let each chain evolve. This code block returns three variables that we will use in the cells that follow:

* `n_samples_unmasked::Int`: number of independent SA chains in this task. Default 10.
* `unmasked_initial::Matrix{Float64}`: shape $4096 \times$ `n_samples_unmasked`, one warm-start state per column (a stored portrait plus a small Gaussian kick).
* `unmasked_samples::Matrix{Float64}`: shape $4096 \times$ `n_samples_unmasked`, one generated face per column after 3000 Langevin steps.

In [ ]:
# ── Task 1: unconditional Stochastic Attention generation ─────────────────
# Run `n_samples_unmasked` Langevin chains starting from random stored portraits
# and let each chain run for n_steps. At low β the chain mixes between memories
# rather than collapsing onto one of them.
n_samples_unmasked, unmasked_initial, unmasked_samples = let

    # how many parallel chains to run. Each becomes one column of the output matrix.
    n_samples_unmasked = 10;

    # Build the warm-start matrix EXPLICITLY so we can show "start vs final" later.
    # `sa_initial_states` returns a (d, n) matrix where each column is a random
    # stored portrait plus a small Gaussian kick of size sa_gen.noise_scale.
    unmasked_initial = sa_initial_states(sa_gen, n_samples_unmasked);

    # STUDENT TASK: call `stochastic_attention_sample` on `sa_gen`, draw
    # `n_samples_unmasked` chains, run for n_steps = 3000, starting from
    # `unmasked_initial`. Assign the result to `unmasked_samples`.
    unmasked_samples = nothing; # TODO: replace `nothing` with the call described above.

    # return -
    n_samples_unmasked, unmasked_initial, unmasked_samples;
end;

Let's see how a single SA chain evolves over 3000 Langevin steps. The figure below snapshots one chain at four points along its trajectory: the warm-start state, after a 2000-step burn-in, a few hundred steps later, and the final collected sample (with one noise-free read-out).

In [ ]:
# ── Visualization: time evolution of a single SA chain ─────────────────────
# We pull one chain (chain_idx = 1) out of the warm-start matrix and run it
# forward in three stages, capturing intermediate states. This shows the
# trajectory: noisy start → burn-in → more steps → final denoised read-out.
let

    # initialize -
    chain_idx = 1                                    # which chain to follow (1..n_samples_unmasked)
    sₒ      = unmasked_initial[:, chain_idx:chain_idx]  # keep as (d, 1) column matrix, not a vector

    # Run the chain in stages so we can snapshot intermediate states.
    # `denoise = false` means we keep the noisy Langevin state; `true` means
    # we strip the final noise term and return the underlying signal.
    s_burn  = stochastic_attention_sample(sa_gen, 1; n_steps = 2000, sₒ = sₒ,     denoise = false)  # after burn-in
    s_more  = stochastic_attention_sample(sa_gen, 1; n_steps =  500, sₒ = s_burn, denoise = false)  # +500 more steps
    s_final = stochastic_attention_sample(sa_gen, 1; n_steps =  500, sₒ = s_more, denoise = true)   # +500 more, denoised read-out

    # Lay out the snapshots side-by-side as a 1×4 grid: start, burn-in, more, final.
    cols   = [sₒ, s_burn, s_more, s_final]
    titles = ["start", "+2000", "+2500", "+3000 (final)"]
    plots  = []
    for (col, t) in zip(cols, titles)
        # un-flatten 4096-vector → 64×64 image, clamp to [0, 1] for display
        img = Gray.(clamp.(reshape(col[:, 1], 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = t, titlefontsize = 8, aspect_ratio = :equal))
    end
    plot(plots...; layout = (1, 4), size = (640, 180), margin = 0Plots.PlotMeasures.mm)
end

__What to look for.__ Read the four tiles left to right as a single chain's trajectory.

* __Start.__ The leftmost tile is the warm-start state: a randomly chosen stored portrait with a small Gaussian perturbation added. It should be recognizable as one of the stored faces.
* __Burn-in (after 2000 steps).__ After 2000 Langevin steps, the score-function term has pulled the state toward a Boltzmann mode while the noise term has let it sample around that mode. If the SA hyperparameters are well-tuned, this tile should already look face-like, with features that are not identical to the start.
* __Drift (after 2500 steps).__ A few hundred more steps confirm that the chain keeps moving and is not stuck on a single attractor; this tile should differ noticeably from the previous one.
* __Final (after 3000 steps).__ We run one last update with the noise turned off (`denoise = true`) and keep the result as the collected sample. This step blends the chain with its closest stored portraits, cleans up the leftover Gaussian fuzz from the noisy steps, and gives us a sharp final image.

If all four tiles look identical (or all four are the same blurry "mean face"), the chain has collapsed: raise `noise_scale` or $\beta$ in Setup so the chain explores distinct basins.

Pick a chain index between $1$ and `n_samples_unmasked`. The cell below stores the picked value in `chain_index::Int` and then shows the starting state (a random stored portrait plus a small Gaussian kick) next to where the chain ended up after `n_steps = 3000` updates. A healthy chain settles between portraits rather than snapping back to its starting image; try a few different `chain_index` values to see typical motion.

In [ ]:
# Pick one of the 10 unconditional chains to inspect more closely.
chain_index = let
    chain_index = 1; # TODO: pick a chain index in 1:n_samples_unmasked
    @assert 1 <= chain_index <= n_samples_unmasked  # guard against typos
    chain_index;
end
println("unconditional · chain $chain_index")

# Show start vs final for the picked chain. Top row of the row-pair model:
# left = warm-start input, right = SA sample after 3000 steps.
let
    cols   = [unmasked_initial[:, chain_index], unmasked_samples[:, chain_index]]
    titles = ["start", "final"]
    plots  = []
    for (col, t) in zip(cols, titles)
        img = Gray.(clamp.(reshape(col, 64, 64), 0.0, 1.0))   # 4096-vec → 64×64 image
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = t, titlefontsize = 8, aspect_ratio = :equal))
    end
    plot(plots...; layout = (1, 2), size = (320, 180), margin = 0Plots.PlotMeasures.mm)
end

For each generated sample, we compute the distance to its nearest stored portrait under the $l_2$ norm, then compare the median to the typical pairwise distance between stored portraits. Three regimes are possible:

* If the *nearest-stored* median is much less than the *pairwise* median, the sampler is producing copies of stored portraits: raise $\sigma$ or lower $\beta$.
* If the two medians are comparable, generated samples sit at typical inter-portrait distance from stored memories: they are novel faces.
* If the *nearest-stored* median is much greater than the *pairwise* median, the chains have wandered off the data manifold: too much noise.

The cell below computes both medians and prints them.

In [ ]:
# ── Novelty diagnostic: how close are SA samples to the nearest stored portrait? ──
# We compare two distributions:
#   • median nearest-stored distance  — for each SA sample, distance to the closest stored portrait
#   • median pairwise distance        — typical distance between two distinct stored portraits
# If the first is much smaller than the second, the chain is regurgitating memories.
# If the two are comparable, the chain is generating novel blends.
let
    nearest_dist = Float64[]
    for k in 1:size(unmasked_samples, 2)
        s = unmasked_samples[:, k]
        # min over all stored portraits — distance from sample k to its nearest memory.
        push!(nearest_dist, minimum(norm(s .- X[:, j]) for j in 1:size(X, 2)))
    end

    # Reference scale: typical pairwise distance among stored portraits.
    # Cap the number of pairs at 50×50 to keep this fast for any n_subjects.
    Nstored = size(X, 2)
    pairs_n = min(50, Nstored)
    pairwise_sample = [norm(X[:, i] .- X[:, j]) for i in 1:pairs_n for j in (i + 1):pairs_n]

    println("median nearest-stored distance for SA samples: $(round(median(nearest_dist), digits=3))")
    println("median pairwise distance among stored portraits: $(round(median(pairwise_sample), digits=3))")
end

___

## Task 2: Subject-conditional generation by masking the softmax
In Task 1 the SA sampler could settle into any of the 100 stored portraits, so the generated faces were a mix of all 10 subjects. In this task we want a *specific* person, i.e., _give me a portrait of subject_ $x$  without retraining the model. This is what we call __conditional generation__.

The trick to __conditional generation__ is that at every step the chain decides which stored memory to blend toward by running a soft competition (a softmax) over all the stored portraits. If we silence every off-subject portrait before that competition runs, the chain has nowhere to go but the in-subject portraits. The chain itself is unchanged; we only edit which memories are allowed to compete. 

Let's look at two strategies for conditional generation, both of which modify the attention softmax:

> __Hard mask vs. log-multiplicity bias:__
>
> Let $\mathbf{x}_j$ denote the $j$-th stored portrait, that is, the $j$-th column of $\mathbf{X}$. At each step of the chain, the softmax competition is driven by the per-memory logit $\beta\,\mathbf{x}_j^{\top}\mathbf{s}$, and both strategies condition on the target subject by editing that logit. The first strategy, the __hard mask__, simply sets the logit of every off-subject memory to $-\infty$,
> $$\mathrm{logit}_{j} \;=\; \beta\,\mathbf{x}_{j}^{\top}\mathbf{s} \;+\; \begin{cases} 0 & \text{if }y_{j} = \text{target}, \\ -\infty & \text{otherwise},\end{cases}$$
> so the exponential sends every off-subject entry to zero and the softmax weight on memory $j$ collapses to a normalized in-subject mass,
> $$p_j \;=\; \frac{\exp(\beta\,\mathbf{x}_j^{\top}\mathbf{s})\,\mathbb{1}[y_j = \text{target}]}{\sum_{k\,:\,y_k = \text{target}}\exp(\beta\,\mathbf{x}_k^{\top}\mathbf{s})},$$
> with the chain confined to the in-subject portraits. The second strategy, the __log-multiplicity bias__ of [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115), replaces the $-\infty$ block with a finite multiplicity-ratio shift $\log\rho$ added only to in-subject logits,
> $$\mathrm{logit}_{j} \;=\; \beta\,\mathbf{x}_{j}^{\top}\mathbf{s} \;+\; \begin{cases} \log\rho & \text{if }y_{j} = \text{target}, \\ 0 & \text{otherwise},\end{cases}$$
> with multiplicity ratio $\rho \geq 1$. The corresponding softmax weight, with the log-sum-exp denominator written out, becomes
> $$p_j \;=\; \frac{\exp(\beta\,\mathbf{x}_j^{\top}\mathbf{s})\,\rho^{\mathbb{1}[y_j = \text{target}]}}{\sum_{k}\exp(\beta\,\mathbf{x}_k^{\top}\mathbf{s})\,\rho^{\mathbb{1}[y_k = \text{target}]}}.$$
> Adding $\log\rho$ to the in-subject log-weights is equivalent to multiplying the in-subject softmax mass by $\rho$ before re-normalization, the multiplicity-ratio conditioning of [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115), Eq. 3; the soft version approaches the hard mask as $\rho \to \infty$ and reduces to the unconditional sampler at $\rho = 1$. In between, even at very large $\rho$, the soft version can fail to reach the hard-mask hit rate: the **calibration gap**.

The two cells below set up the conditioning, then run the masked sampler. They bind five names for use in the rest of Task 2:

* `target_class::Int`: which subject id to generate, between $0$ and `n_subjects` $-\,1$. Default 2.
* `n_samples_class::Int`: how many chains to run for the target subject. Default 10.
* `class_mask::Vector{Bool}` of length $N$: `class_mask[j] = (y[j] == target_class)`. `true` means "memory $j$ is allowed", `false` means "blocked".

No model retraining, no rebuilding: the mask is a single `Vector{Bool}` passed at sample time.

In [ ]:
# ── Task 2a: hard-mask subject-conditional sampling — pick a target subject ──
# A "hard mask" is a boolean vector telling SA which memories are visible at
# sampling time. By masking out everything except one subject's portraits, the
# chain can only attend to in-class memories — so at high β it locks onto that
# subject's identity.
target_class, n_samples_class, class_mask = let

    # initialize -
    target_class = 2;          # TODO: which subject id to generate (0..n_subjects-1)
    n_samples_class = 10;      # TODO: how many samples to draw for this subject

    # STUDENT TASK: build a Bool vector `class_mask` of length size(X, 2) where
    # entry j is true iff y[j] == target_class. A comprehension is the cleanest
    # one-liner.
    class_mask = nothing; # TODO: replace with the comprehension described above.

    println("class_mask leaves ", count(class_mask), " of ", length(class_mask), " stored portraits visible")

    # return -
    target_class, n_samples_class, class_mask;
end;

The code block starts with a random face drawn from the target subject's portraits (whose id lives in `target_class::Int`, with a small Gaussian nudge added), then runs the masked sampler for 3000 steps. The returned variables are:

* `class_initial::Matrix{Float64}`: shape $4096 \times$ `n_samples_class`, warm-start states drawn only from in-subject memories (because we pass `hard_mask = class_mask`).
* `class_samples::Matrix{Float64}`: shape $4096 \times$ `n_samples_class`, one generated portrait per column after 3000 masked Langevin steps.

What should we expect to see?

In [ ]:
# ── Task 2a: run the hard-masked chains ──────────────────────────────────
# STUDENT TASK: Uncomment the block below to run the hard-masked SA chains.
# Builds the warm-start matrix EXPLICITLY (drawn only from in-subject memories
# because we pass `hard_mask = class_mask`) so we can show start vs final per chain.

class_initial, class_samples = let

    # placeholders so the cell runs even before you uncomment.
    class_initial = nothing;
    class_samples = nothing;

    #==
    # `sa_initial_states` with a hard_mask draws warm-starts ONLY from in-class
    # memories. So column k of class_initial is "in-class memory + small noise".
    class_initial = sa_initial_states(sa_cond, n_samples_class; hard_mask = class_mask);

    # Run the conditional chains. `hard_mask` is forwarded to the dynamics, so
    # at every step attention is restricted to in-class memories. At β_cond high
    # the softmax is sharp enough that the chain locks onto a single stored
    # portrait of `target_class`.
    class_samples = stochastic_attention_sample(sa_cond, n_samples_class;
        n_steps = 3000, hard_mask = class_mask, sₒ = class_initial);
    ==#

    # return -
    class_initial, class_samples;
end;

__What should we expect to see?__ The hard mask gives subject-conditional samples at near-perfect hit rate. 

In [ ]:
# ── Visualize Task 2a: stored portraits vs hard-mask SA samples ──────────
# Top row: actual stored portraits OF the target subject.
# Bottom row: SA samples drawn under the hard mask.
# At high β the bottom row should look like specific stored portraits (Hebbian
# recall) — i.e. each SA sample matches some top-row image closely.
let

    # initialize -
    n_show = 5  # how many tiles per row (1..n_samples_class)

    # First n_show stored portraits OF the target subject (top row).
    target_idx = findall(==(target_class), y)
    stored_target = X[:, target_idx[1:min(n_show, length(target_idx))]]

    # First n_show SA samples (bottom row).
    sa_target = class_samples[:, 1:n_show]

    # Build the plot list row-by-row: top row first, then bottom row.
    plots = []
    for i ∈ 1:n_show
        img = Gray.(clamp.(reshape(stored_target[:, i], 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = "S$(target_class) stored $i", titlefontsize = 7, aspect_ratio = :equal))
    end

    for i ∈ 1:n_show
        img = Gray.(clamp.(reshape(sa_target[:, i], 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = "S$(target_class) SA $i", titlefontsize = 7, aspect_ratio = :equal))
    end

    # Render: 2 rows × n_show columns. Plots fill row-major in the order pushed.
    plot(plots...; layout = (2, n_show), size = (140 * n_show, 160 * 2),
        margin = 0Plots.PlotMeasures.mm)
end

Both rows should look like subject `target_class::Int`. The top row is __real portraits of that subject__. The bottom row is hard-mask SA samples. The hard mask _blocks every off-subject memory_ exactly, so SA cannot drift to a different person. Stylistic variation across the bottom row (different shots blended) is fine and expected.

### Where do we start and where do we end up?
Next, let's dig into this a little bit more by looking at where we start, and where we end up, for a single chain, i.e., a single run of the masked SA sampler. Pick a chain index between $1$ and `n_samples_class::Int` (this is the chain we are going to inspect). The cell below stores the picked value in `class_chain_index::Int`, and an [AssertionError will be raised](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) if it falls outside the valid range, in which case you should pick a different `class_chain_index::Int`. 

In [ ]:
# Pick one of the n_samples_class hard-mask chains to inspect more closely.
class_chain_index = let

    # initialize -
    class_chain_index = 9; # TODO: pick a chain index in 1:n_samples_class
    @assert 1 <= class_chain_index <= n_samples_class # guard against typos
    println("subject $target_class · chain $class_chain_index")
    class_chain_index;
end;

__Let's look at the what we give and what we get__: The cell below shows some random starting face (with some small Gaussian corruption) next to where the chain ended up after 3000 __masked__ Langevin steps. Because every off-subject memory has logit $-\infty$, both tiles should be the requested subject: the final tile differs from the start only because the chain has mixed across in-subject portraits.

In [ ]:
# Start vs final for the picked class-conditional chain.
# Left = in-class warm-start memory + noise; right = SA sample after 3000 steps
# under the hard mask. At β_cond = 8 the right tile should snap to a stored
# portrait of `target_class`.
let

    # initialize -
    cols   = [class_initial[:, class_chain_index], class_samples[:, class_chain_index]]
    titles = ["start", "final"]
    plots  = [] # storage to hold the plot objects

    # build one tile per (column, title) pair
    for (col, t) ∈ zip(cols, titles)
        img = Gray.(clamp.(reshape(col, 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = t, titlefontsize = 8, aspect_ratio = :equal))
    end
    plot(plots...; layout = (1, 2), size = (320, 180), margin = 0Plots.PlotMeasures.mm)
end

### Fidelity check

Next, let's check the __fidelity__ of the generated samples. The big question is: do the generated faces look like the target subject? We answer this question with a training-free classifier that, for each sample, computes the $l_2$ distance to every per-subject centroid and assigns the sample to the nearest one. This is a nearest-centroid classifier (small $l_2$ distance indicates high similarity).

If the closest stored portrait belongs to the target subject class, we count that as a hit; otherwise, it's a miss. The centroids `class_means::Dict{Int, Vector{Float64}}` were precomputed in the Setup section.

The cell below loops over all `n_subjects` subjects, draws `n_samples_class` samples each under the hard mask, and reports the fraction that the classifier maps back to the requested subject. Results land in `hardmask_table::DataFrame`, with one row per subject and columns `target::Int`, `hits::Int`, `total::Int`, `hit_rate::Float64`.

In [ ]:
# ── Task 2a evaluation: hard-mask hit-rate sweep over all subjects ────────
# STUDENT TASK: Uncomment the block below to run the hit-rate sweep.
# For each subject c, run the hard-masked sampler with mask = (y .== c), then
# classify each sample by the nearest-class-mean rule. A "hit" is when the
# classifier label matches the target subject c. We expect hit_rate ≈ 1.0 for
# all subjects when β_cond is high.

hardmask_table = let

    # initialize -
    df = DataFrame(target = Int[], hits = Int[], total = Int[], hit_rate = Float64[]);

    #==
    # Loop over all subjects, draw n_samples_class samples per subject under the
    # subject-specific hard mask, and evaluate with the nearest-centroid classifier.
    for c ∈ 0:(n_subjects - 1)
        m = [yⱼ == c for yⱼ ∈ y]                                                  # mask for subject c
        samples = stochastic_attention_sample(sa_cond, n_samples_class;
            n_steps = 3000, hard_mask = m)
        labels = [classify_by_nearest_mean(samples[:, k], class_means) for k ∈ 1:n_samples_class]
        h = count(==(c), labels)                                                  # hits = correctly classified samples
        push!(df, (target = c, hits = h, total = n_samples_class, hit_rate = h / n_samples_class))
    end

    # pretty-print the table
    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact)
    );
    ==#

    df; # return the DataFrame so it can be used downstream
end;

### Hard mask vs. log-multiplicity bias: the calibration gap

A hard mask zeros off-subject logits exactly. The _log-multiplicity bias_ of [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115), Eq. 3, instead adds the finite shift $\log\rho$ to in-subject logits (with $\rho \geq 1$ the multiplicity ratio) and leaves off-subject logits at $0$. As $\rho \to \infty$ the soft version approaches the hard mask, but the interpolation does not always land where we expect in pixel space: that gap is the **calibration gap**.

The cell below sweeps $\rho$ and stores the ratio-versus-hit-rate table in `softbias_table::DataFrame`, with one row per `(ρ, target_class)` pair and columns `ρ::Float64`, `log_ρ::Float64`, `target::Int`, `hits::Int`, `total::Int`, `hit_rate::Float64`.

In [ ]:
# ── Task 2b: log-multiplicity (soft-bias) sweep ───────────────────────────
# STUDENT TASK: Uncomment the block below to run the ρ sweep.
# Instead of hard-masking out non-target memories, we ADD log(ρ) to the logits
# of in-class memories. This is a soft preference, not a hard restriction. As
# ρ → ∞ the soft bias dominates and the sampler approaches the hard-mask limit;
# at ρ = 1 the soft bias vanishes and we recover the unconditional sampler.

softbias_table = let

    # ρ: multiplicity ratio. ρ = 1 is the unconditional sampler;
    # ρ → ∞ approaches the hard mask. The in-class logit shift is log(ρ).
    # We use sa_soft (β_soft ≪ β_cond) so log(ρ) is competitive with β·X'·s.
    df = DataFrame(ρ = Float64[], log_ρ = Float64[], target = Int[], hits = Int[],
                   total = Int[], hit_rate = Float64[])

    #==
    in_class = [yj == target_class for yj in y]

    # Sweep ρ from 1 (no bias) up to 10000 (very strong bias, near hard-mask limit).
    for ρ in [1.0, 10.0, 100.0, 1000.0, 10000.0]
        # soft-bias vector: log(ρ) on in-class memories, 0 on out-of-class.
        sb_vec = Float64.([m ? log(ρ) : 0.0 for m in in_class])

        # Run n_samples_class chains under this soft bias and classify them.
        samples = stochastic_attention_sample(sa_soft, n_samples_class;
            n_steps = 3000, soft_bias = sb_vec)
        labels = [classify_by_nearest_mean(samples[:, k], class_means) for k in 1:n_samples_class]
        h = count(==(target_class), labels)

        push!(df, (ρ = ρ, log_ρ = log(ρ), target = target_class,
                   hits = h, total = n_samples_class, hit_rate = h / n_samples_class))
    end

    # pretty-print the table
    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact)
    );
    ==#

    df; # return -
end;

In [ ]:
# ── Visualize Task 2b: ρ sweep on a fixed warm-start ─────────────────────
# Layout: one row per ρ value, columns = | start | chain 1 | chain 2 | chain 3 |.
# All chains in all rows share a SINGLE fixed warm-start (the first stored portrait
# of target_class) so the row-to-row comparison isolates the effect of ρ on the
# trajectory — same starting point, only the soft bias changes.
# Uses sa_soft (β = β_soft) so log(ρ) is in the regime where it can steer the softmax.
let
    ρ_values = [1.0, 10.0, 100.0, 1000.0, 10000.0]
    n_per    = 3                                            # chains per row
    in_class = [yj == target_class for yj in y]

    # Fixed warm-start: first in-class stored portrait, broadcast to all n_per chains.
    j_seed     = findfirst(in_class)                        # column index in X of the first in-class portrait
    seed_state = X[:, j_seed]                               # the actual portrait (a 4096-vector)
    sₒ_mat     = repeat(seed_state, 1, n_per)               # (d, n_per) matrix — same column n_per times
    start_img  = Gray.(clamp.(reshape(seed_state, 64, 64), 0.0, 1.0))   # pre-rendered start image, reused on every row

    plots = []
    for (i, ρ) in enumerate(ρ_values)
        # Build the soft-bias logit shift: log(ρ) on in-class memories, 0 elsewhere.
        sb_vec  = Float64.([m ? log(ρ) : 0.0 for m in in_class])
        # Run n_per chains from the same warm-start under this soft bias.
        samples = stochastic_attention_sample(sa_soft, n_per;
            n_steps = 3000, soft_bias = sb_vec, sₒ = sₒ_mat)

        # column 0: warm-start image. Same picture in every row, but we re-emit it
        # so the plot grid has a consistent left "start" column. Title only on row 1
        # to avoid title clutter; ylabel labels the row with its ρ value.
        push!(plots, heatmap(start_img;
            color = :grays, axis = false, ticks = false, colorbar = false,
            title = i == 1 ? "start" : "", titlefontsize = 9,
            ylabel = "ρ = $(ρ)", ylabelfontsize = 9,
            aspect_ratio = :equal))

        # columns 1..n_per: chain end states for this ρ.
        for k in 1:n_per
            img = Gray.(clamp.(reshape(samples[:, k], 64, 64), 0.0, 1.0))
            push!(plots, heatmap(img;
                color = :grays, axis = false, ticks = false, colorbar = false,
                title = i == 1 ? "chain $k" : "", titlefontsize = 9,
                aspect_ratio = :equal))
        end
    end

    # Render: length(ρ_values) rows × (n_per + 1) columns. Order pushed = row-major.
    plot(plots...; layout = (length(ρ_values), n_per + 1),
        plot_title = "Log-multiplicity sweep at β_soft = $(sa_soft.β), target_class = $(target_class) (start = stored portrait $(j_seed))",
        plot_titlefontsize = 10,
        size = (170 * (n_per + 1), 160 * length(ρ_values)),
        left_margin = 6Plots.PlotMeasures.mm,
        margin = 1Plots.PlotMeasures.mm)
end

___

## Task 3: $\beta$-sweep, the Hebbian/Hopfield bridge under a mask

In this task, we measure the bridge between the modern-Hopfield update and the H-Mem read step we covered in week 15. The intro asserted that at low $\beta$ the stochastic attention masked sampler returns soft blends of stored memories, while at high $\beta$ it collapses to a single stored column (the Hebbian limit). 

Sweeping $\beta \equiv \beta_{\mathrm{cond}}$ inside the masked sampler should explore this narrative: the transition from soft blend to hard recall is the bridge between modern Hopfield and H-Mem.

> __Experiment design.__
> We hold the subject mask, chain length, and noise settings ($\eta, \sigma$) fixed at the Task 2 values. The only knob we sweep is $\beta$, across a logarithmic grid from very soft to very sharp. For each $\beta$ we build a [fresh `MyStochasticAttentionModel`](src/Types.jl) over the same memory bank $\mathbf{X}$, run masked Langevin chains for the target subject, and compute two diagnostics:
>
> * **Hit rate via nearest-mean classifier**, the fidelity score from Task 2. The hard mask blocks every off-subject memory regardless of $\beta$, so hit rate should stay flat across the sweep.
> * **Median nearest-stored distance** $\min_{j \in \{1,\dots,M\}} \|\mathbf{s} - \mathbf{x}_{j}\|_{2}$, where $\mathbf{x}_{j}$ is the $j$-th stored memory (column of $\mathbf{X}$) and $M$ is the number of stored memories; this is the novelty proxy from Task 1. At low $\beta$ this stays moderate (the chain settles near a convex blend of stored portraits); at high $\beta$ it falls toward $0$ (the chain collapses onto a stored column).
> 
> __What should we expect to see?__ Hit rate stays flat across the sweep: the mask, not $\beta$, enforces conditioning. The median nearest-stored distance falls smoothly from soft blend at low $\beta$ to near-replica at high $\beta$. That descent is the bridge.

The cell below runs the sweep and binds four names:

* `β_grid::Vector{Float64}`: the logarithmic grid of inverse temperatures we sweep over.
* `bridge_table::DataFrame`: one row per $\beta$, columns `β::Float64`, `hit_rate::Float64`, `median_nearest_stored::Float64`.
* `bridge_samples_low::Matrix{Float64}`: shape $4096 \times$ `n_samples_class`, the SA samples at the smallest $\beta$ in the grid (the "soft blend" extreme).
* `bridge_samples_high::Matrix{Float64}`: shape $4096 \times$ `n_samples_class`, the SA samples at the largest $\beta$ (the "Hebbian recall" extreme).

In [ ]:
# ── DQ3: β sweep — bridge novelty (low β) to recall (high β) ─────────────
# STUDENT TASK: Uncomment the block below to run the β-sweep.
# We hold the hard mask fixed and sweep only β. For each β:
#   1. build a fresh SA model with this β
#   2. draw n_samples_class hard-masked samples (same mask = class_mask)
#   3. compute hit_rate (correctly-classified rate via nearest-class-mean)
#   4. compute median_nearest_stored — the regime witness for novelty
# We also stash the lowest-β and highest-β sample matrices so DQ3 figures
# can show the two ends of the bridge.

β_grid, bridge_table, bridge_samples_low, bridge_samples_high = let

    # initialize -
    β_grid = [0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 4.0, 8.0, 32.0, 128.0]; # TODO: try a denser or wider grid
    df = DataFrame(β = Float64[], hit_rate = Float64[], median_nearest_stored = Float64[])
    bridge_samples_low  = nothing  # filled in on the first iteration
    bridge_samples_high = nothing  # filled in on the last  iteration

    #==
    # Sweep β with everything else fixed (mask, step_size, noise_scale, n_steps).
    for (i, β) ∈ enumerate(β_grid)
        # Build a per-β model. Cheap because building just stores hyperparameters.
        sa_β = build(MyStochasticAttentionModel, (
            memories     = X,
            labels       = y,
            β            = β,
            step_size    = sa_cond.step_size,   # reuse the Task 2a step/noise so β is the only knob moving
            noise_scale  = sa_cond.noise_scale,
        ))

        # Hard-masked chains for this β.
        samples = stochastic_attention_sample(sa_β, n_samples_class;
            n_steps = 3000, hard_mask = class_mask)

        # hit_rate: nearest-class-mean classifier hits on target_class.
        labels = [classify_by_nearest_mean(samples[:, k], class_means) for k in 1:n_samples_class]
        hit_rate = count(==(target_class), labels) / n_samples_class

        # novelty witness: median over chains of "distance to closest stored portrait".
        nearest = [minimum(norm(samples[:, k] .- X[:, j]) for j in 1:size(X, 2))
                   for k in 1:n_samples_class]

        push!(df, (β = β, hit_rate = hit_rate, median_nearest_stored = median(nearest)))

        # Stash sample matrices at the two ends of the sweep for the DQ3 figures.
        i == 1                && (bridge_samples_low  = samples)
        i == length(β_grid)   && (bridge_samples_high = samples)
    end

    # pretty-print the table
    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact)
    );
    ==#

    # return -
    β_grid, df, bridge_samples_low, bridge_samples_high;
end;

In [ ]:
# ── DQ3 plot: novelty (median nearest-stored distance) vs β ──────────────
# As β increases, attention sharpens and the chain locks onto stored memories,
# so median_nearest_stored → 0 (perfect recall). As β decreases, the chain
# blends memories, so median_nearest_stored grows (novel outputs).
# Hit rate is flat at 1.0 across the sweep (the mask, not β, enforces conditioning),
# so we only plot the novelty axis.
let
    plot(bridge_table.β, bridge_table.median_nearest_stored;
        xscale = :log10, marker = :circle, lw = 2,                       # log scale on β so the wide range fits
        xlabel = "β (log scale)", ylabel = "median nearest-stored",
        label = "subject $(target_class)",
        title = "Novelty vs β",
        titlefontsize = 10, labelfontsize = 9, tickfontsize = 8, legendfontsize = 8,
        size = (600, 380),
        left_margin = 6Plots.PlotMeasures.mm,
        bottom_margin = 6Plots.PlotMeasures.mm,
        top_margin = 3Plots.PlotMeasures.mm,
        right_margin = 3Plots.PlotMeasures.mm)
end

__What to look for at a low $\beta$ value.__ **Initialization (warm start).** Each of the $K=5$ chains starts at its own state $\mathbf{s}_0^{(k)} = \mathbf{x}_{j_k} + 0.05\cdot\boldsymbol{\xi}_k$, where $j_k$ is a *uniformly random* in-subject column of $\mathbf{X}$ (any of subject $\mathrm{target\_class}$'s stored portraits) and $\boldsymbol{\xi}_k\sim\mathcal{N}(0,\mathbf{I})$ is light Gaussian noise. The five chains thus begin at five *different* in-subject portraits, each lightly jittered. **The top row in the figure below shows these actual chain inputs** $\mathbf{s}_0^{(k)}$, they look like jittered photos of subject $\mathrm{target\_class}$. Bottom row: the same five chains after $T=3000$ Langevin steps. 

Despite starting at five *different* portraits, all bottom-row samples should look similar to one another (they all converged to the same in-subject mean) and should appear as *smoothed, ghosted averages* of subject $\mathrm{target\_class}$'s portraits, recognizable as that subject but not as any one specific photo. 

Features that survive averaging (overall facial structure, lighting direction shared across in-subject portraits) stay; features that vary across portraits (specific expression, exact pose) blur out. This is the modern-Hopfield / soft-blend regime, the opposite extreme of the high-$\beta$ Hebbian recall figure below.

In [ ]:
# ── DQ3 figure (low β end): warm-start vs SA sample ──────────────────────
# Top row: actual warm-start inputs (in-class memory + small noise).
# Bottom row: the SA samples those starts produced after 3000 steps at β = β_grid[1].
# At very low β attention is nearly uniform — the chain blends memories, so the
# bottom row looks like a soft averaged "ghost" of the in-class portraits.
let
    n_show = 5
    β_low  = β_grid[1]                                                      # smallest β in the sweep

    # Build a fresh model at β_low using the same step/noise as Task 2a, so
    # β is the only thing differing from the high-β case.
    sa_β   = build(MyStochasticAttentionModel, (
        memories    = X,
        labels      = y,
        β           = β_low,
        step_size   = sa_cond.step_size,
        noise_scale = sa_cond.noise_scale,
    ))

    # Warm-starts drawn ONLY from in-class memories (because of class_mask).
    s0_warm = sa_initial_states(sa_β, n_show; hard_mask = class_mask)

    # Run the chains from those warm-starts under the same hard mask.
    sa_low  = stochastic_attention_sample(sa_β, n_show;
        n_steps = 3000, hard_mask = class_mask, sₒ = s0_warm)

    # Build a 2×n_show grid: row 1 = warm-starts, row 2 = SA samples.
    plots = []
    for i in 1:n_show
        img = Gray.(clamp.(reshape(s0_warm[:, i], 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = "s₀ warm #$i", titlefontsize = 7, aspect_ratio = :equal))
    end
    for i in 1:n_show
        img = Gray.(clamp.(reshape(sa_low[:, i], 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = "SA β=$(β_low) #$i", titlefontsize = 7, aspect_ratio = :equal))
    end
    plot(plots...; layout = (2, n_show), size = (140 * n_show, 160 * 2),
        margin = 0Plots.PlotMeasures.mm)
end

__What to look for at a high $\beta$ value.__ Each chain begins from a pure Gaussian noise vector $\mathbf{s}_0$ (top row), no information about $\mathbf{X}$ was used to initialize a chain. After 3000 Langevin steps with the hard mask, the chain has snapped onto a stored in-subject memory (middle row), and the bottom row shows that memory $\mathbf{x}_j$ explicitly with its column index $j$. 

Top vs. middle row: noise → face. Middle vs. bottom row: the chain has produced a verbatim copy of $\mathbf{x}_j$, the **Hebbian recall** limit. Repeated $j$ values across columns mean multiple chains landed in the same basin; distinct $j$ values mean the chains found different in-subject attractors.

In [ ]:
# ── DQ3 figure (high β end): cold-start, 3-row story ─────────────────────
# Cold-start = start the chain from pure Gaussian noise (no information about X).
# Restructured as a 3-row story:
#   row 1 — the random Gaussian initial state s₀ (looks like static)
#   row 2 — the SA sample after 3000 Langevin steps at β = β_grid[end]
#   row 3 — the in-subject stored memory each sample is closest to (the attractor it landed on)
# Each column is one chain: noise → final sample → matched memory. Because β
# is large and the mask restricts attention to in-class memories, the chain
# lands on one of the stored target-class portraits — Hebbian recall from noise.
let
    Random.seed!(42) # reset the RNG so the figure is reproducible, even after running the previous cells
    n_show = 5
    β_high = β_grid[end]                                                    # largest β in the sweep

    # Fresh model at β_high (everything else matches Task 2a).
    sa_β   = build(MyStochasticAttentionModel, (
        memories    = X,
        labels      = y,
        β           = β_high,
        step_size   = sa_cond.step_size,
        noise_scale = sa_cond.noise_scale,
    ))

    # Cold-start: pure Gaussian noise, no peek at X. d = pixel dimension (4096).
    d = size(X, 1)
    s0_cold = randn(d, n_show)

    # Run the chains under the hard mask. Even from noise, sharp attention +
    # mask should pull the chain to an in-class stored memory.
    sa_cold = stochastic_attention_sample(sa_β, n_show;
        n_steps = 3000, hard_mask = class_mask, sₒ = s0_cold)

    # For each final sample, find the closest stored memory across ALL of X
    # (not just in-class) — this is which attractor the chain landed on.
    nearest_idx = [argmin([norm(sa_cold[:, k] .- X[:, j]) for j in 1:size(X, 2)])
                   for k in 1:n_show]

    # Build a 3×n_show grid: row 1 = s₀ noise, row 2 = SA sample, row 3 = nearest stored.
    plots = []
    for i in 1:n_show
        img = Gray.(clamp.(reshape(s0_cold[:, i], 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = "s₀ cold #$i", titlefontsize = 7, aspect_ratio = :equal))
    end
    for i in 1:n_show
        img = Gray.(clamp.(reshape(sa_cold[:, i], 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = "SA β=$(β_high) #$i", titlefontsize = 7, aspect_ratio = :equal))
    end
    for i in 1:n_show
        j = nearest_idx[i]
        img = Gray.(clamp.(reshape(X[:, j], 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false, colorbar = false,
            title = "nearest X[:, $j]", titlefontsize = 7, aspect_ratio = :equal))
    end
    plot(plots...; layout = (3, n_show), size = (140 * n_show, 160 * 3),
        margin = 0Plots.PlotMeasures.mm)
end

___

## Discussion

Answer each question in the comment block below it, then set the corresponding `did_I_answer_DQ*` flag to `true`.

**DQ1: Three regimes of Stochastic Attention.** Stochastic Attention has three knobs (the inverse temperature $\beta$, the step size $\eta$, and the noise scale $\sigma$). With $\eta = 1$ and fixed $\sigma = 0.10$, the chain's behavior is dominated by $\beta$. 

To make the regime contrast visible on raw Olivetti pixels, [the `try_sa(...) helper function`](#) builds its model on the **centered** memory bank $\mathbf{X} - \bar{\mathbf{x}}$ (with $\bar{\mathbf{x}}$ added back for display); without centering, the strong pixel-space cross-correlations cause every chain to collapse to a single _winner_ column at sharp $\beta$ regardless of where it started (attention collapse). The three scaffolded calls below sweep $\beta$ on the centered model to expose three qualitatively different regimes: 
* **A (replication)** at $\beta = 64$, where the softmax is effectively one-hot and each chain locks onto its own starting basin (median nearest-stored $\approx 0$); 
* **B (transition)** at $\beta = 0.1$, where the softmax is partially soft and chains drift off their start memory into a sharp blend of a few in-class portraits (median nearest-stored $\approx 1$, sample norm still moderate); and 
* **C (full blend)** at $\beta = 0.005$, where the softmax is nearly uniform and the chain collapses to the centered global mean (sample norm $\approx 0$, displayed as a smooth ghost-average face).

Run the three cells, then answer:

1. Describe what you see in each regime in one sentence ("face-like," "averaged," "smoothed," etc. are legitimate observations).
2. Explain each regime in terms of the SA energy landscape: what does $\beta$ do to the softmax weights at high vs. low values? Why does Regime A produce a stored portrait verbatim while Regime C produces a smooth average?
3. With $\eta = 1$, each Langevin step is $\mathbf{s}_{t+1} = \mathbf{X}\cdot\mathrm{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s}_t) + \sigma\cdot\boldsymbol{\xi}_t$. Why does $\sigma$ have so little effect on this dataset, and in what regime would it start to matter?

> __Strategy:__
> Run all three cells and eyeball the differences, using the `med_near` value in each plot title to ground your description. Think of $\beta$ as controlling softmax sharpness: high $\beta$ → one column dominates (Regime A), low $\beta$ → weights spread across all columns so $\mathbf{X}\cdot\mathrm{softmax}(\ldots)$ collapses to the mean (Regime C). For question 3, note that with $\eta = 1$ each step overwrites the state, and the noise's inner product with any stored memory is small in expectation, so the softmax at step $t+2$ essentially ignores $\sigma\cdot\boldsymbol{\xi}_t$; $\sigma$ would only matter if $\eta < 1$ (the chain integrates state across steps) or if inter-memory distances were comparable to $\sigma$.

In [ ]:
# DQ1 scaffold: defined here in the notebook (not in src/). Once you run this cell,
# `try_sa` is available in the notebook's session scope and can be called from any
# subsequent cell. It is meant for quick parameter exploration — for production code
# you would move a function like this into src/Compute.jl and export it.

"""
    try_sa(β, η, σ; n_show = 8, n_steps = 3000)

Build a fresh unconditional `MyStochasticAttentionModel` over the **centered**
memory bank `X .- mean(X; dims=2)` with the given inverse temperature `β`, step
size `η`, and noise scale `σ`, run `n_show` independent Langevin chains for
`n_steps`, and display a 2-row grid:

- **Row 1 (starts):** each chain's warm-start state — a randomly chosen stored
  portrait + a small Gaussian kick.
- **Row 2 (finals):** the same chain's state after `n_steps` Langevin updates.

Comparing the two rows column-by-column shows what each chain did with its
starting prompt: lock onto a stored portrait (Regime A), drift to a sharp blend
(Regime B), or collapse to the global mean (Regime C).

### Why centering?
On raw Olivetti pixels, every face has a strong DC component (mean pixel ≈ 0.56)
and pairwise inner products ⟨X[:,i], X[:,j]⟩ are nearly as large as the diagonals
‖X[:,j]‖². At sharp β, that pushes the softmax onto a single "winner" column
for *any* state — every chain collapses to the same memory regardless of where
it started (attention collapse). Subtracting the per-pixel mean kills the DC
term and lets each chain lock onto its own starting basin at high β. The mean
is added back at display time so tiles stay on the [0,1] grayscale.

### Arguments
- `β::Real`: inverse temperature for the softmax.
- `η::Real`: Langevin step size in (0, 1].
- `σ::Real`: Langevin noise scale (≥ 0).

### Keyword arguments
- `n_show::Int = 8`: number of independent chains.
- `n_steps::Int = 3000`: number of Langevin iterations per chain.

### Returns
A `Plots.Plot` object: 2 × `n_show` grid of 64×64 grayscale heatmaps. Title
shows (β, η, σ) and `med_near = median_k min_j ‖sₖ − Xc[:,j]‖`.
"""
function try_sa(β::Real, η::Real, σ::Real; n_show::Int = 8, n_steps::Int = 3000)

    # ── Step 1: center the memory bank ────────────────────────────────────────
    # We compute the per-pixel mean across all 100 stored faces. Subtracting it
    # gives Xc, a memory bank where every column has mean 0. This kills the DC
    # component that otherwise dominates inner products on Olivetti and causes
    # "attention collapse" at sharp β. We keep μ around to add back later when
    # we display tiles, so the rendered images still live on the [0,1] grayscale.
    μ  = mean(X; dims = 2)              # shape (d, 1) — per-pixel mean over the 100 columns of X
    Xc = X .- μ                         # shape (d, M) — centered memory bank used by the dynamics
    d, M = size(Xc)                     # d = 4096 (= 64×64 pixels), M = 100 (10 subjects × 10 photos)

    # ── Step 2: build a fresh SA model on the centered bank ───────────────────
    # `build` is the project's factory for MyStochasticAttentionModel. We pass in
    # the centered memories Xc (NOT raw X), the labels y, and the three knobs:
    #   β = inverse temperature (sharpness of the softmax)
    #   η = step size (how much of the modern-Hopfield update to apply per step)
    #   σ = noise scale (Gaussian noise added to each Langevin step)
    sa = build(MyStochasticAttentionModel, (
        memories    = Xc,
        labels      = y,
        β           = Float64(β),
        step_size   = Float64(η),
        noise_scale = Float64(σ),
    ))

    # ── Step 3: build the warm-start matrix S0 EXPLICITLY ─────────────────────
    # Normally `stochastic_attention_sample` would internally call
    # `sa_initial_states` to pick warm-start states. We replicate that recipe
    # here (pick a random stored column per chain, add a 5%-amplitude Gaussian
    # kick) so we can *display* the starts in row 1 of the figure. Without this,
    # the chain inputs would be invisible to students reading the plot.
    rng = Random.GLOBAL_RNG
    S0  = zeros(Float64, d, n_show)     # one column per chain — d-dimensional state
    for k in 1:n_show
        j = rand(rng, 1:M)              # pick a random stored portrait index for chain k
        S0[:, k] = Xc[:, j] .+ 0.05 .* randn(rng, d)  # centered portrait + small noise
    end

    # ── Step 4: run the Langevin chains ───────────────────────────────────────
    # We pass S0 explicitly via `sₒ = S0` so the displayed starts match what the
    # chains actually see. Each chain runs for `n_steps` Langevin updates of the
    # form  s_{t+1} = (1−η)·s_t + η·X·softmax(β·X^T·s_t) + σ·ξ_t.
    samples = stochastic_attention_sample(sa, n_show; n_steps = n_steps, sₒ = S0)

    # ── Step 5: compute the regime witness `med_near` ─────────────────────────
    # For each chain, find the nearest stored memory in centered space and
    # measure that distance. The median across chains is our scalar regime
    # diagnostic — it goes 0 (chains landed ON stored portraits, Regime A)
    # → small (chains landed near a blend of a few memories, Regime B)
    # → ≈‖Xc[:,j]‖ (chains landed at the centered mean = origin, Regime C).
    nearest = [minimum(norm(samples[:, k] .- Xc[:, j]) for j in 1:M)
               for k in 1:n_show]
    med_near = median(nearest)

    # ── Step 6: build the 2-row display ───────────────────────────────────────
    # Each chain becomes a column. Top row = where the chain started, bottom row
    # = where it ended. Reading top→bottom in any column tells you what the
    # chain DID with its starting prompt at this β.
    plots = []

    # Row 1: chain starts. We add μ back so the centered residual reads as a face.
    # `clamp.(..., 0, 1)` prevents the small noise kick from pushing pixels off
    # the grayscale range — without it the renderer would saturate and look ugly.
    for i in 1:n_show
        img = Gray.(clamp.(reshape(S0[:, i] .+ vec(μ), 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false,
            colorbar = false, title = "start #$i", titlefontsize = 7,
            aspect_ratio = :equal))
    end

    # Row 2: chain finals — same column k as row 1, so column k tells one chain's story.
    for i in 1:n_show
        img = Gray.(clamp.(reshape(samples[:, i] .+ vec(μ), 64, 64), 0.0, 1.0))
        push!(plots, heatmap(img; color = :grays, axis = false, ticks = false,
            colorbar = false, title = "final #$i", titlefontsize = 7,
            aspect_ratio = :equal))
    end

    # 2 rows × n_show cols. Title prints the knobs plus med_near so the regime is
    # legible from the figure title alone, even before the eye reads the tiles.
    plot(plots...; layout = (2, n_show), size = (140 * n_show, 320),
        plot_title = "β=$(β), η=$(η), σ=$(σ)   med_near=$(round(med_near; digits=2))",
        plot_titlefontsize = 9,
        margin = 0Plots.PlotMeasures.mm)
end;

In [ ]:
# ── DQ1 — Regime A (replication) ──────────────────────────────────────────
# At β = 64 the softmax β·Xc^T·s is so sharp that it concentrates almost all
# its mass on a single column of Xc. The update s ← Xc·softmax(...) therefore
# returns essentially that one stored memory (in centered coordinates). The
# σ = 0.10 noise is small relative to inter-memory distances in centered space,
# so chains can't escape the basin they land in.
#
# What you should see:
#   • Row 1 (starts):  8 different noisy stored portraits.
#   • Row 2 (finals):  8 sharp, contrasty stored portraits.
#   • Most columns have top ≈ bottom (chain locked onto its starting basin),
#     though 1–2 chains may drift to a different in-class memory because
#     centered face-face cross-correlations aren't perfectly zero.
#   • Title shows  med_near ≈ 0.0  — every chain ended ON a stored memory.
# try_sa(64.0, 1.0, 0.10) # TODO: uncomment to run

In [ ]:
# ── DQ1 — Regime B (transition / sharp blend) ─────────────────────────────
# At β = 0.1 the softmax is partially soft: it concentrates on a few centered
# columns rather than one, so Xc·softmax(...) is a *sharp blend* of those
# memories — face-shaped but no longer a verbatim stored portrait. Note: the
# transition band shifts with centering (centered ‖Xc[:,j]‖² is ~20× smaller
# than the raw value), so β = 0.1 is the right "mid" knob here even though it
# would land in the ghost-mean regime on the raw memory bank.
#
# What you should see:
#   • Row 1 (starts):  8 different noisy stored portraits.
#   • Row 2 (finals):  blurry, face-shaped tiles that don't match their starts.
#   • Tiles in row 2 look like soft mixtures of a couple of in-class faces,
#     not a single sharp portrait and not the ghost average.
#   • Title shows  med_near ≈ 1–2  — chains landed near, but not on, stored memories.
# try_sa(0.1, 1.0, 0.10) # TODO: uncomment to run

In [ ]:
# ── DQ1 — Regime C (full blend / ghost mean) ──────────────────────────────
# At β = 0.005 the softmax weights are nearly uniform over all 100 columns of
# Xc. The update Xc·softmax(...) ≈ mean of Xc columns ≈ 0 (because Xc is
# centered). After we add μ back at display time, the rendered tile is the
# global-mean face — the average of all 100 stored portraits.
#
# What you should see:
#   • Row 1 (starts):  8 different noisy stored portraits.
#   • Row 2 (finals):  8 nearly-identical, smooth, low-contrast "ghost average" faces.
#   • The chain has forgotten its starting basin entirely; identity is washed out.
#   • Title shows  med_near ≈ 4.5  — chains landed far from any stored memory,
#     near the centered origin.
# try_sa(0.005, 1.0, 0.10) # TODO: uncomment to run

In [ ]:
#= Put your answer to DQ1 here. =#

In [ ]:
did_I_answer_DQ1 = false; # TODO: set to `true` after answering DQ1

**DQ2: Why does the log-multiplicity bias under-perform a hard mask?** A boolean mask reaches near-perfect subject-conditional hit rate, but the multiplicity-ratio shift $\log\rho$ on in-subject logits often does not, even at $\rho = 10^4$. Explain the calibration gap.

> __Strategy:__
>
> Compare the hard-mask and log-multiplicity hit-rate tables side by side. For small $\rho$ (small $\log\rho$), why does the hit rate barely move? Think about the magnitude of the inner-product term $\beta\cdot\mathbf{X}^{\top}\mathbf{s}$ versus the additive shift $\log\rho$ on a 4096-dimensional state. For large $\rho$, why does the soft version still miss? Connect this to the **calibration gap** of [Varner (2026, arXiv:2603.20115)](https://arxiv.org/abs/2603.20115): conditioning is exact at the level of the sampler's softmax weights, but the decoded sample can disagree because the inner-product term steers the chain toward whichever attractor is closest, regardless of subject.

We've printed the hard-mask and soft-bias hit-rate tables side by side for comparison. 

In [ ]:
# DQ2 scaffold: print the hard-mask hit rate for the target subject alongside the
# log-multiplicity sweep, so the two regimes sit next to each other.
let
    hard_target = filter(:target => ==(target_class), hardmask_table)
    println("Hard mask, subject $(target_class):")
    pretty_table(hard_target;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

In [ ]:
let
    println("Log-multiplicity bias, subject $(target_class), ρ sweep:")
    pretty_table(softbias_table;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

In [ ]:
#= Put your answer to DQ2 here. =#

In [ ]:
did_I_answer_DQ2 = false; # TODO: set to `true` after answering DQ2

**DQ3: What does the $\beta$-sweep recover empirically?** Across the $\beta$-grid, hit rate stays flat but median nearest-stored distance falls smoothly. Explain what this curve is and where the transition between the two regimes happens.

> __Strategy:__
>
> The Task 3 sweep cell returns `bridge_table::DataFrame` with one row per $\beta$ in `β_grid` and columns `β`, `hit_rate`, `median_nearest_stored`. Read its rows from low $\beta$ to high $\beta$ and answer two questions:
> * (1) Why is `hit_rate` flat even at $\beta = 0.005$? What is actually doing the conditioning in the masked sampler?
> * (2) What does the descent in `median_nearest_stored` recover empirically? Compare the low-$\beta$ and high-$\beta$ image grids (`bridge_samples_low` vs. `bridge_samples_high`): the low end should show face-like blends of multiple in-subject portraits (modern-Hopfield mean); the high end should show near-replicas of stored portraits ([L15](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026/tree/main/lectures/week-15) H-Mem read step).

Identify where on the grid the transition happens and describe the regime in the middle.

In [ ]:
#= Put your answer to DQ3 here. =#

In [ ]:
did_I_answer_DQ3 = false; # TODO: set to `true` after answering DQ3

___

## Optional extensions
This was a proof-of-concept exploration of masked Stochastic Attention on the Olivetti faces. The three papers below sketch where the idea goes next.

### Different applications
* __Small protein families.__ A small alignment of <100 sequences from a single Pfam family is the regime where deep generative models overfit, but where Stochastic Attention has linear-in-alignment-size cost on a laptop. See [Varner, *Training-Free Generation of Protein Sequences from Small Family Alignments via Stochastic Attention*, arXiv:2603.14717](https://arxiv.org/abs/2603.14717).
* __Conditioning on a binding screen.__ The log-multiplicity bias of Task 2 is the recipe; see [Varner, *Conditioning Protein Generation via Hopfield Pattern Multiplicity*, arXiv:2603.20115](https://arxiv.org/abs/2603.20115). The same paper diagnoses the calibration-gap phenomenon we saw in DQ2.
* __Synthetic patient cohorts.__ Replacing "stored protein sequence" with "stored longitudinal patient record" lets SA augment small clinical cohorts of <30 patients in domains like pregnancy and rare disease without retraining. See [Varner, Bravo, McBride, Orfeo, Bernstein, *Validated Synthetic Patient Generation for Small Longitudinal Cohorts*, arXiv:2604.07557](https://arxiv.org/abs/2604.07557).

### Other things to try
* __Causal mask:__ instead of masking by class, set up a causal mask (`mask[j] = (j ≤ t)` at time step `t`) so the chain only attends to earlier memories.
* __Per-pattern temperatures:__ replace the scalar $\beta$ with a vector of per-pattern temperatures, decoupling sampler confidence from mask membership.
* __Compare to the spiking H-Mem of [L15b](https://github.com/varnerlab/CHEME-5820-Labs-Spring-2026/tree/main/labs/week-15/L15b):__ sketch how to turn the hard-mask SA into a one-shot recall network with dynamically-written memory.

___

## Summary

This practicum used Stochastic Attention with a masked softmax to generate subject-conditional faces from a small Olivetti memory bank without any model training.

> __Key Takeaways:__
>
> * **Training-free face generation:** Injecting Gaussian noise into the modern-Hopfield update produces a Langevin sampler whose stationary distribution is the Hopfield Boltzmann measure over stored memories. The sampler runs on a laptop in linear time with no gradient descent and no tuned weights.
> * **Hard mask vs. log-multiplicity bias:** A boolean mask over the attention softmax delivers subject-conditional generation by excluding off-subject memories. The additive log-multiplicity shift on in-subject logits (Varner 2026, arXiv:2603.20115, Eq. 3) rarely overcomes the inner-product term, leaving a calibration gap between attention-space conditioning and pixel-space output.
> * **Temperature sweep recovers the Hebbian bridge:** Holding the mask and noise fixed, hit rate stays flat across the temperature sweep but the median nearest-stored distance falls smoothly from soft blend to near-replica. The descent recovers the modern-Hopfield to Hebbian transition empirically inside the masked sampler.

The masked sampler shows that conditioning, novelty, and recall are controlled by simple, separate knobs on a Hopfield memory.
___

## Tests
In the code block below, we check some values in your notebook and give you feedback on which items are correct or different. Unhide the code block (if you are curious) to see how the tests are set up.

In [ ]:
let
    @testset verbose = true "CHEME 5820 Practicum S2026" begin

        @testset "Setup, Data, and Prerequisites" begin
            @test _DID_INCLUDE_FILE_GET_CALLED == true
            @test isnothing(n_subjects) == false
            @test isnothing(images_per_subject) == false
            @test isnothing(β_gen) == false
            @test isnothing(β_cond) == false
            @test isnothing(β_soft) == false
            @test β_gen > 0
            @test β_cond > 0
            @test β_soft > 0
            @test size(X, 1) == 4096
            @test size(X, 2) == n_subjects * images_per_subject
            @test length(y) == size(X, 2)
            @test sort(unique(y)) == collect(0:(n_subjects - 1))
            @test isnothing(sa_gen) == false
            @test sa_gen.β == β_gen
            @test isnothing(sa_cond) == false
            @test sa_cond.β == β_cond
            @test isnothing(sa_soft) == false
            @test sa_soft.β == β_soft
            @test sa_gen.step_size > 0
            @test sa_gen.noise_scale >= 0
            @test isnothing(class_means) == false
            @test length(class_means) == n_subjects
        end

        @testset "Task 1: Stochastic Attention sampling on faces" begin
            @test size(unmasked_initial) == (4096, n_samples_unmasked)
            @test size(unmasked_samples) == (4096, n_samples_unmasked)
            @test all(isfinite, unmasked_initial)
            @test all(isfinite, unmasked_samples)
            @test isnothing(chain_index) == false
            @test 1 <= chain_index <= n_samples_unmasked
            @test did_I_answer_DQ1 == true
        end

        @testset "Task 2: Masked SA = subject-conditional generation" begin
            @test 0 <= target_class <= n_subjects - 1
            @test length(class_mask) == size(X, 2)
            @test count(class_mask) == count(==(target_class), y)
            @test size(class_initial) == (4096, n_samples_class)
            @test all(isfinite, class_initial)
            @test size(class_samples) == (4096, n_samples_class)
            @test isnothing(class_chain_index) == false
            @test 1 <= class_chain_index <= n_samples_class
            @test mean(hardmask_table.hit_rate) > 0.7
            # Calibration gap: at moderate ρ (log_ρ ≤ 5, i.e. ρ ≲ 150) the soft-bias
            # under-performs the hard mask. At very large ρ the gap closes, so we only
            # test the regime where the gap is visible.
            @test mean(hardmask_table.hit_rate) > maximum(filter(:log_ρ => <=(5.0), softbias_table).hit_rate)
            @test did_I_answer_DQ2 == true
        end

        @testset "Task 3: β-sweep recovers the Hebbian/Hopfield bridge" begin
            @test length(β_grid) >= 4
            @test issorted(β_grid)
            @test all(>(0), β_grid)
            @test nrow(bridge_table) == length(β_grid)
            @test all(isfinite, bridge_table.hit_rate)
            @test all(isfinite, bridge_table.median_nearest_stored)
            @test mean(bridge_table.hit_rate) > 0.7
            # bridge claim: novelty (distance to nearest stored) at the
            # smallest β should exceed novelty at the largest β: the
            # Hebbian-limit collapse onto stored portraits.
            @test bridge_table.median_nearest_stored[1] > bridge_table.median_nearest_stored[end]
            @test size(bridge_samples_low)  == (4096, n_samples_class)
            @test size(bridge_samples_high) == (4096, n_samples_class)
            @test did_I_answer_DQ3 == true
        end
    end
end;

___